"""
Week 6 Deliverable: Feature Engineering and Market Metrics
IDX Exchange Data Analyst Internship
 
Engineers the key market metrics on top of the Week 4-5 cleaned (and
school-district-enriched) sold and listing datasets:
 
- PriceRatio               = ClosePrice / OriginalListPrice
- PricePerSqFt              = ClosePrice / LivingArea
- DaysOnMarket              = raw field (already present, restated here)
- Year / Month / YrMo       = derived from CloseDate
- CloseToOriginalListRatio  = ClosePrice / OriginalListPrice
- ListingToContractDays     = PurchaseContractDate - ListingContractDate
- ContractToCloseDays       = CloseDate - PurchaseContractDate
 
Also produces at least one segmented summary table grouped by
PropertyType/PropertySubType and CountyOrParish/MLSAreaMajor.
"""

In [1]:
import pandas as pd
 
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

In [2]:
# PART 1 - SOLD DATASET
# =================================================================
print("=" * 70)
print("FEATURE ENGINEERING: SOLD  (sold_cleaned.csv)")
print("=" * 70)
 
df_sold = pd.read_csv('sold_cleaned.csv', low_memory=False)
rows_before_sold = len(df_sold)
cols_before_sold = len(df_sold.columns)
print(f"\nLoaded {rows_before_sold} rows, {cols_before_sold} columns")

FEATURE ENGINEERING: SOLD  (sold_cleaned.csv)

Loaded 448091 rows, 90 columns


In [3]:
#Make sure the date fields are datetime (they were saved as text in the CSV)

df_sold['CloseDate'] = pd.to_datetime(df_sold['CloseDate'], errors='coerce')
df_sold['ListingContractDate'] = pd.to_datetime(df_sold['ListingContractDate'], errors='coerce')
df_sold['PurchaseContractDate'] = pd.to_datetime(df_sold['PurchaseContractDate'], errors='coerce')

In [17]:
# --- Price Ratio ---
df_sold['PriceRatio'] = df_sold['ClosePrice'] / df_sold['OriginalListPrice']

# --- Price Per Sq Ft ---
df_sold['PricePerSqFt'] = df_sold['ClosePrice'] / df_sold['LivingArea']
 
# --- Days on Market (raw field, already present -- just confirming it's there) ---
if 'DaysOnMarket' not in df_sold.columns:
    print("  Warning: DaysOnMarket column not found.")
 
# --- Year / Month / YrMo, derived from CloseDate ---
df_sold['Year'] = df_sold['CloseDate'].dt.year
df_sold['Month'] = df_sold['CloseDate'].dt.month
df_sold['YrMo'] = df_sold['CloseDate'].dt.to_period('M').astype(str)
 
# --- Close to Original List Ratio ---
df_sold['CloseToOriginalListRatio'] = df_sold['ClosePrice'] / df_sold['OriginalListPrice']
 
# --- Listing to Contract Days ---
df_sold['ListingToContractDays'] = (df_sold['PurchaseContractDate'] - df_sold['ListingContractDate']).dt.days

#--- Contract to Close Days ---
df_sold['ContractToCloseDays'] = (df_sold['CloseDate'] - df_sold['PurchaseContractDate']).dt.days
 
print("\n--- Engineered metrics summary (sold) ---")
metric_cols_sold = ['PriceRatio', 'PricePerSqFt', 'DaysOnMarket', 'Year', 'Month', 'YrMo',
                     'CloseToOriginalListRatio', 'ListingToContractDays', 'ContractToCloseDays']
print(df_sold[metric_cols_sold].describe())


--- Engineered metrics summary (sold) ---
         PriceRatio  PricePerSqFt   DaysOnMarket           Year          Month  CloseToOriginalListRatio  ListingToContractDays  ContractToCloseDays
count  4.472680e+05  4.478360e+05  448091.000000  448091.000000  448091.000000              4.472680e+05          447892.000000        447893.000000
mean            inf           inf      37.311151    2024.782116       6.048593                       inf              45.158773            31.607853
std             NaN           NaN      53.612576       0.752301       3.239866                       NaN              86.215690            60.395951
min    0.000000e+00  0.000000e+00    -288.000000    2024.000000       1.000000              0.000000e+00          -36407.000000          -331.000000
25%    9.537211e-01  3.680625e+02       8.000000    2024.000000       3.000000              9.537211e-01              10.000000            21.000000
50%    9.956522e-01  5.374033e+02      18.000000    2025.000000

/Users/ashley/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/ashley/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/ashley/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [14]:
print("\n--- Sample output table (sold) ---")
sample_cols_sold = ['ListingKey', 'ClosePrice', 'OriginalListPrice', 'LivingArea',
                     'PriceRatio', 'PricePerSqFt', 'YrMo',
                     'ListingToContractDays', 'ContractToCloseDays']
sample_cols_sold = [c for c in sample_cols_sold if c in df_sold.columns]
print(df_sold[sample_cols_sold].head())


--- Sample output table (sold) ---
   ListingKey  ClosePrice  OriginalListPrice  LivingArea  PriceRatio  PricePerSqFt     YrMo  ListingToContractDays  ContractToCloseDays
0   551985747    240000.0           499000.0      1140.0    0.480962    210.526316  2024-01                  777.0                 65.0
1   522107581    815000.0           759900.0      1974.0    1.072510    412.867275  2024-01                  114.0                919.0
2   510919001    810000.0           739900.0      1974.0    1.094743    410.334347  2024-01                  255.0                778.0
3  1079166779    858000.0                NaN      1995.0         NaN    430.075188  2024-01                  188.0               -188.0
4  1075037759   1890500.0          1890500.0      3194.0    1.000000    591.891046  2024-01                    0.0                  0.0


In [5]:
# --- Segment Analysis: by PropertyType/PropertySubType ---
print("\n--- Segment summary: by PropertySubType (sold) ---")
if 'PropertySubType' in df_sold.columns:
    segment_subtype_sold = df_sold.groupby('PropertySubType').agg(
        n_records=('ClosePrice', 'count'),
        median_close_price=('ClosePrice', 'median'),
        median_ppsf=('PricePerSqFt', 'median'),
        median_days_on_market=('DaysOnMarket', 'median'),
        median_price_ratio=('PriceRatio', 'median'),
    ).sort_values('n_records', ascending=False)
    print(segment_subtype_sold)
else:
    print("  PropertySubType column not found, skipping.")


--- Segment summary: by PropertySubType (sold) ---
                       n_records  median_close_price  median_ppsf  median_days_on_market  median_price_ratio
PropertySubType                                                                                             
SingleFamilyResidence     335674            895987.5   533.333333                   17.0            1.000000
Condominium                73612            626747.0   564.111083                   24.0            0.985679
Townhouse                  26245            800000.0   560.698197                   17.0            1.000000
ManufacturedOnLand          5796            320000.0   225.107754                   30.0            0.969900
Duplex                      2493            910000.0   543.333333                   21.0            0.983333
StockCooperative            1765            360000.0   396.205357                   20.0            0.985185
Cabin                        507            241000.0   287.934783           

In [6]:
# --- Segment Analysis: by CountyOrParish ---
print("\n--- Segment summary: by CountyOrParish (sold) ---")
if 'CountyOrParish' in df_sold.columns:
    segment_county_sold = df_sold.groupby('CountyOrParish').agg(
        n_records=('ClosePrice', 'count'),
        median_close_price=('ClosePrice', 'median'),
        median_ppsf=('PricePerSqFt', 'median'),
        median_days_on_market=('DaysOnMarket', 'median'),
    ).sort_values('median_close_price', ascending=False)
    print(segment_county_sold)
else:
    print("  CountyOrParish column not found, skipping.")
 
df_sold.to_csv('sold_features.csv', index=False, encoding='utf-8')
print(f"\nSaved: sold_features.csv ({len(df_sold)} rows, {len(df_sold.columns)} columns)")


--- Segment summary: by CountyOrParish (sold) ---
                 n_records  median_close_price  median_ppsf  median_days_on_market
CountyOrParish                                                                    
Del Norte                1           2485000.0   390.907661                  320.0
Other County             4           2462500.0   242.673993                   22.0
San Mateo             7800           1700000.0  1052.631579                   12.0
Santa Clara          19648           1600000.0   965.774998                   10.0
San Francisco         1029           1200000.0   897.368421                   16.0
...                    ...                 ...          ...                    ...
Imperial               316            306000.0   211.585831                   39.0
Modoc                    2            282000.0   104.858398                   46.5
Sierra                   1            255000.0   222.902098                    1.0
Foreign Country          4          

In [7]:
# PART 2 - LISTING DATASET
# =================================================================
print("\n" + "=" * 70)
print("FEATURE ENGINEERING: LISTING  (listing_cleaned.csv)")
print("=" * 70)
 
df_listing = pd.read_csv('listing_cleaned.csv', low_memory=False)
rows_before_listing = len(df_listing)
cols_before_listing = len(df_listing.columns)
print(f"\nLoaded {rows_before_listing} rows, {cols_before_listing} columns")
 
df_listing['CloseDate'] = pd.to_datetime(df_listing['CloseDate'], errors='coerce')
df_listing['ListingContractDate'] = pd.to_datetime(df_listing['ListingContractDate'], errors='coerce')
df_listing['PurchaseContractDate'] = pd.to_datetime(df_listing['PurchaseContractDate'], errors='coerce')


FEATURE ENGINEERING: LISTING  (listing_cleaned.csv)

Loaded 614541 rows, 81 columns


In [8]:
# --- Price Ratio ---
df_listing['PriceRatio'] = df_listing['ClosePrice'] / df_listing['OriginalListPrice']
 
# --- Price Per Sq Ft ---
df_listing['PricePerSqFt'] = df_listing['ClosePrice'] / df_listing['LivingArea']
 
if 'DaysOnMarket' not in df_listing.columns:
    print("  Warning: DaysOnMarket column not found.")
 
# --- Year / Month / YrMo ---
# NOTE: for listing records that haven't closed, CloseDate will be null and
# these will be NaN/NaT -- that's expected, not a bug.
df_listing['Year'] = df_listing['CloseDate'].dt.year
df_listing['Month'] = df_listing['CloseDate'].dt.month
df_listing['YrMo'] = df_listing['CloseDate'].dt.to_period('M').astype(str)

In [9]:
# --- Close to Original List Ratio ---
df_listing['CloseToOriginalListRatio'] = df_listing['ClosePrice'] / df_listing['OriginalListPrice']
 
# --- Listing to Contract Days ---
df_listing['ListingToContractDays'] = (df_listing['PurchaseContractDate'] - df_listing['ListingContractDate']).dt.days
 
# --- Contract to Close Days ---
df_listing['ContractToCloseDays'] = (df_listing['CloseDate'] - df_listing['PurchaseContractDate']).dt.days
 
print("\n--- Engineered metrics summary (listing) ---")
metric_cols_listing = ['PriceRatio', 'PricePerSqFt', 'DaysOnMarket', 'Year', 'Month', 'YrMo',
                        'CloseToOriginalListRatio', 'ListingToContractDays', 'ContractToCloseDays']
print(df_listing[metric_cols_listing].describe())
 
print("\n--- Sample output table (listing) ---")


--- Engineered metrics summary (listing) ---
         PriceRatio  PricePerSqFt   DaysOnMarket           Year          Month  CloseToOriginalListRatio  ListingToContractDays  ContractToCloseDays
count  1.501860e+05  1.509740e+05  614541.000000  174574.000000  174574.000000              1.501860e+05          291792.000000        174543.000000
mean            inf           inf      18.748129    2024.889239       6.504056                       inf              22.534483            26.540176
std             NaN           NaN      25.683527       0.727415       3.516706                       NaN              27.743083            17.404738
min    6.672227e-04  3.741981e-01     -58.000000    2024.000000       1.000000              6.672227e-04            -731.000000          -169.000000
25%    9.687484e-01  3.966420e+02       5.000000    2024.000000       3.000000              9.687484e-01               7.000000            18.000000
50%    1.000000e+00  5.660377e+02      11.000000    2025.000

/Users/ashley/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/ashley/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/ashley/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [11]:
sample_cols_listing = ['ListingKey' if 'ListingKey' in df_listing.columns else 'ListingId',
                        'ListPrice', 'OriginalListPrice', 'LivingArea',
                        'PriceRatio', 'PricePerSqFt', 'YrMo',
                        'ListingToContractDays', 'ContractToCloseDays']
sample_cols_listing = [c for c in sample_cols_listing if c in df_listing.columns]
print(df_listing[sample_cols_listing].head())


   ListingKey   ListPrice  OriginalListPrice  LivingArea  PriceRatio  PricePerSqFt YrMo  ListingToContractDays  ContractToCloseDays
0  1074973329   1340000.0          1340000.0      1301.0         NaN           NaN  NaT                  127.0                  NaN
1  1074954552   2500000.0          2500000.0      2788.0         NaN           NaN  NaT                    NaN                  NaN
2  1074936537   3150000.0          3150000.0      3250.0         NaN           NaN  NaT                    NaN                  NaN
3  1074917818   3090000.0          3090000.0      7456.0         NaN           NaN  NaT                    NaN                  NaN
4  1074143166  12725000.0         12725000.0      2321.0         NaN           NaN  NaT                    NaN                  NaN


In [12]:
print("\n--- Segment summary: by PropertySubType (listing) ---")
if 'PropertySubType' in df_listing.columns:
    segment_subtype_listing = df_listing.groupby('PropertySubType').agg(
        n_records=('ListPrice', 'count'),
        median_list_price=('ListPrice', 'median'),
        median_ppsf=('PricePerSqFt', 'median'),
        median_days_on_market=('DaysOnMarket', 'median'),
    ).sort_values('n_records', ascending=False)
    print(segment_subtype_listing)
else:
    print("  PropertySubType column not found, skipping.")


--- Segment summary: by PropertySubType (listing) ---
                       n_records  median_list_price  median_ppsf  median_days_on_market
PropertySubType                                                                        
SingleFamilyResidence     447634           925000.0   568.757555                   10.0
Condominium               111356           639900.0   575.949367                   12.0
Townhouse                  36177           818000.0   568.814056                   12.0
ManufacturedOnLand          8496           329994.5   228.023201                    9.0
Duplex                      3900           975000.0   569.685934                    9.0
StockCooperative            1910           375000.0   398.936170                   10.0
Cabin                       1135           299999.0   296.089385                    7.0
Triplex                      788          1200000.0   474.345550                    8.0
MixedUse                     571           849900.0   391.236307 

In [13]:
print("\n--- Segment summary: by CountyOrParish (listing) ---")
if 'CountyOrParish' in df_listing.columns:
    segment_county_listing = df_listing.groupby('CountyOrParish').agg(
        n_records=('ListPrice', 'count'),
        median_list_price=('ListPrice', 'median'),
        median_days_on_market=('DaysOnMarket', 'median'),
    ).sort_values('median_list_price', ascending=False)
    print(segment_county_listing)
else:
    print("  CountyOrParish column not found, skipping.")
 
df_listing.to_csv('listing_features.csv', index=False, encoding='utf-8')
print(f"\nSaved: listing_features.csv ({len(df_listing)} rows, {len(df_listing.columns)} columns)")


--- Segment summary: by CountyOrParish (listing) ---
                n_records  median_list_price  median_days_on_market
CountyOrParish                                                     
San Mateo           11239          1648000.0                   11.0
Santa Clara         29674          1499840.0                   10.0
Santa Cruz           5206          1250000.0                   13.0
Marin                 259          1249000.0                   13.0
Orange              58975          1222800.0                   10.0
...                   ...                ...                    ...
Tehama                653           349000.0                    8.0
Lake                 3185           345000.0                    8.0
Imperial              507           315000.0                    7.0
Lassen                 36           235750.0                    7.5
Modoc                  13           225000.0                    3.0

[63 rows x 3 columns]

Saved: listing_features.csv (614541 ro